# Stage 01 - Data Ingestion: Parsing Tabular Data (CSV & Excel)
### Lecture 15: Converting Rows to Semantic Documents for RAG

This notebook implements tabular data ingestion for vector search:
1. Parsing CSV data using `CSVLoader` and `pandas`.
2. Parsing multi-sheet Excel workbooks (`.xlsx`) via `openpyxl`.
3. Transforming rows into natural language narrative sentences for high-quality embedding capture.
4. Preserving columnar metadata (SKU, price, category, timestamps).


In [1]:
# Step 1: Environment Setup and Tabular Asset Discovery
from pathlib import Path
import pandas as pd
from langchain_community.document_loaders import CSVLoader
from langchain_core.documents import Document

repo_root = Path.cwd()
while not (repo_root / "data").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

csv_path = repo_root / "data" / "tabular" / "products.csv"
excel_path = repo_root / "data" / "tabular" / "inventory.xlsx"

print(f"[CONFIG] CSV Path: {csv_path} (Exists: {csv_path.exists()})")
print(f"[CONFIG] Excel Path: {excel_path} (Exists: {excel_path.exists()})")
assert csv_path.exists() and excel_path.exists(), "Both CSV and Excel sample files must exist."


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_21412\931547617.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


[CONFIG] CSV Path: D:\Code\AI\rag-mastery\data\tabular\products.csv (Exists: True)
[CONFIG] Excel Path: D:\Code\AI\rag-mastery\data\tabular\inventory.xlsx (Exists: True)


In [2]:
# Step 2: Parsing CSV with CSVLoader
csv_loader = CSVLoader(file_path=str(csv_path), encoding="utf-8")
csv_docs = csv_loader.load()

print(f"[INFO] CSVLoader parsed {len(csv_docs)} rows into Documents.")
print(f"Sample Document Content:\n{csv_docs[0].page_content}")
print(f"Sample Metadata: {csv_docs[0].metadata}")

# Assertions
assert len(csv_docs) > 0, "CSVLoader must produce documents from tabular rows."
assert "row" in csv_docs[0].metadata, "Metadata must record the row number."


[INFO] CSVLoader parsed 5 rows into Documents.
Sample Document Content:
product: MacBook Pro 16
category: Laptops
price: 2499
description: High-performance laptop with M3 Max chip and 36GB unified memory.
Sample Metadata: {'source': 'D:\\Code\\AI\\rag-mastery\\data\\tabular\\products.csv', 'row': 0}


In [3]:
# Step 3: Semantic Excel Ingestion with pandas
# Transforming raw spreadsheet rows into natural narrative descriptions for embeddings
df_excel = pd.read_excel(excel_path)
semantic_excel_docs = []

for idx, row in df_excel.iterrows():
    # Construct an embedding-friendly sentence representation
    narrative_repr = " | ".join([f"{col}: {val}" for col, val in row.items()])
    semantic_excel_docs.append(
        Document(
            page_content=narrative_repr,
            metadata={"source": excel_path.name, "row_index": idx}
        )
    )

print(f"[INFO] Processed {len(semantic_excel_docs)} Excel inventory items.")
print(f"Sample Narrative Row: {semantic_excel_docs[0].page_content}")

# Assertions
assert len(semantic_excel_docs) == len(df_excel), "Document count must match Excel row count."
print("[SUCCESS] Tabular data parsing (CSV and Excel) verified.")


[INFO] Processed 5 Excel inventory items.
Sample Narrative Row: product: MacBook Pro 16 | category: Laptops | price: 2499 | description: High-performance laptop with M3 Max chip and 36GB unified memory.
[SUCCESS] Tabular data parsing (CSV and Excel) verified.
